# EDA под кейс

Порядок здесь не случайный: сначала то, что может обесценить всю дальнейшую работу
(утечки, неверная схема валидации), и только потом красивые графики.

**Правило дня:** к концу этого ноутбука у вас должны быть ответы на шесть вопросов —
тип задачи, таргет, метрика, единица предсказания, что известно в момент предсказания,
формат сдачи. Не переходите к признакам, пока они не записаны.

In [ ]:
import sys

sys.path.insert(0, "..")

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

from ml.src.config import TrainConfig
from ml.src.data import leakage_report, load_dataset, split_columns

cfg = TrainConfig()
df = load_dataset(cfg.data_path)
df.shape

## 1. Что в данных

Типы, пропуски, кардинальность — одной таблицей, чтобы не листать десять ячеек.

In [ ]:
profile = pd.DataFrame({
    "тип": df.dtypes.astype(str),
    "пропуски_%": (df.isna().mean() * 100).round(1),
    "уникальных": df.nunique(),
    "пример": df.iloc[0],
})
profile.sort_values("пропуски_%", ascending=False)

## 2. Таргет

Смотрим дисбаланс: при сильном перекосе accuracy бесполезна, нужны ROC-AUC, PR-AUC или F1.
Заодно проверяем, нет ли в таргете пропусков — такие строки в обучение не идут.

In [ ]:
target = cfg.target
print(df[target].describe())
print()
print(df[target].value_counts(normalize=True).round(4))

## 3. Утечки — до всего остального

Признак с очень высокой связью с таргетом почти всегда означает поле, заполняемое
постфактум. Найденная и объяснённая утечка — сильный пункт на защите.

In [ ]:
suspects = leakage_report(df, target)
print("Подозрительные колонки:", suspects or "не найдено")

numeric_cols = df.select_dtypes("number").columns
corr = df[numeric_cols].corr()[target].drop(target).sort_values(key=abs, ascending=False)
corr.head(15)

## 4. Время

Если временная колонка есть, валидация должна быть по времени, а не случайная.
Проверяем диапазон и то, как ведёт себя таргет во времени: резкий сдвиг означает,
что на новых данных модель поведёт себя иначе.

In [ ]:
time_col = cfg.time_column
if time_col and time_col in df.columns:
    dates = pd.to_datetime(df[time_col])
    print("Диапазон:", dates.min(), "—", dates.max())
    by_month = df.assign(_m=dates.dt.to_period("M")).groupby("_m")[target].agg(["mean", "count"])
    display(by_month)
else:
    print("Временной колонки нет — валидация stratified или group")

## 5. Признаки в разрезе таргета

Ищем, что реально разделяет классы: это будущие фичи и будущие слайды.

In [ ]:
numeric, categorical = split_columns(df, target, [cfg.id_column, time_col])

for col in numeric[:8]:
    stats = df.groupby(target)[col].mean()
    if len(stats) == 2 and stats.iloc[0]:
        diff = (stats.iloc[1] / stats.iloc[0] - 1) * 100
        print(f"{col:<24} класс0={stats.iloc[0]:>10.2f}  класс1={stats.iloc[1]:>10.2f}  разница {diff:+.1f}%")

In [ ]:
for col in categorical[:6]:
    rates = df.groupby(col)[target].agg(["mean", "count"]).sort_values("mean", ascending=False)
    print(f"\n{col}:")
    display(rates.head(8))

## 6. Выводы

Запишите здесь то, что пойдёт в презентацию, пока не забылось:

- **Инсайт 1:** ...
- **Инсайт 2:** ...
- **Схема валидации:** ... (и почему именно она)
- **Найденные утечки:** ...
- **Идеи признаков:** ...

Дальше — `python -m ml.src.train`, а новые признаки пишите в `ml/src/features.py`,
чтобы сервис считал их так же, как обучение.